
# Fiducial 밝기 조정

본 측정 전에 fiducial 밝기를 맞출 때 쓴다. fiducial 전용 노출 한 장에서 peak 을 찾아 fiducial ID 를 붙이고,
그림 위에 `ID: peak` 을 적고 (포화는 빨간 고리), fiducial 마다 같은 색 눈금의 crop 을 늘어놓는다. 표에는 peak(창 안 최대 화소, 배경 포함),
flux(배경을 뺀 창 안 합), nsat(포화 화소 수) 이 나온다. `FID_GROUPS` 에 밝기 조정 그룹을 적으면 그룹별 색과 요약표가 나온다.

- peak 이 sat_level(65000) 에 닿으면 포화라 위치가 틀어진다. 측정에 쓸 노출에서는 모든 fiducial 이 포화되지 않으면서 가장 어두운 것도 threshold 보다 충분히 밝아야 한다.
- 포화된 fiducial 끼리는 peak 이 모두 65535 라 비교가 안 되므로 flux 로 본다.
- fiber 가 같이 찍힌 이미지면 `TARGET_FILE` 을 준다 (fiber 와 fiducial 이 같은 격자에 있어 fiducial 30개만으로는 회전이 정해지지 않는다).


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
BROKEN_FIBERS = []              # 끊어져 빛이 안 나오는 fiber 의 홀 ID. 예: ['G4']
FIXED_FIBERS  = []              # 빛은 나오지만 고장나 움직이지 않는 positioner 의 홀 ID. 예: ['C6']
SKY_FIBERS    = ['A6', 'B7', 'B8', 'G0', 'H1', 'I0', 'N7', 'N8', 'O6']   # sky fiber 홀 (켜져 있으면). 꺼져 있으면 []
ANGLE_SIGN    = (1, 1)          # 출력 각도 = 부호 x 계산각 (alpha 반시계 양, beta >= 0)

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN)


In [ ]:

MAGNIFICATION  = 8.40           # camera -> focal 배율 초기값 (2026-10-17 세팅 기준; 몇 % 안으로 맞으면 됨)
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'


In [ ]:

FID_IMAGES  = ['/path/to/fid_2ms.fits', '/path/to/fid_10ms.fits']   # fiducial 전용 노출 (이름이 ...0.fits 로 끝나야 한다)
TARGET_FILE = None                                                    # fiber 도 같이 찍혔으면 object.info 경로
FID_GROUPS  = None                                                    # 예: {'G1': ['A7', 'B1', ...], 'G2': [...], 'G3': [...], 'G4': [...]}
ids, xf, yf = lab.fiducial_positions()
print(len(ids), 'fiducials:', ids)


In [ ]:

results = {}
for f in FID_IMAGES:
    T, S = lab.fiducial_brightness(f, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS, nwindow=NWINDOW,
                                   magnification=MAGNIFICATION, groups=FID_GROUPS, sat_level=SAT_LEVEL, target_file=TARGET_FILE)
    results[os.path.basename(f)] = (T, S)
    if S is not None:
        display(S.round(0))
    display(T[T.id != '?'].round(2))



## 노출 간 비교
같은 fiducial 의 peak 을 노출마다 나란히 (포화되지 않은 노출에서 peak 은 노출 시간에 비례해야 한다).


In [ ]:

cmp = pd.DataFrame({k: T[T.id != '?'].set_index('id').peak for k, (T, _) in results.items()})
if FID_GROUPS:
    cmp.insert(0, 'group', [next((g for g, m in FID_GROUPS.items() if i in m), '') for i in cmp.index])
cmp.round(0)
